Imports & ADLS Config

ABFSS is the native protocol used by Azure Data Lake Storage Gen2. It allows Databricks to securely access files directly from ADLS.

In [0]:
from pyspark.sql.functions import *
from uuid import uuid4

storage_account_name = "hexawaretrainingadl"
container_name = "raw"
storage_account_key = "****"
base_path = (
    f"abfss://{container_name}@"
    f"{storage_account_name}.dfs.core.windows.net/"
)

PIPELINE_RUN_ID = str(uuid4())

In [0]:
%sql
create schema if not exists retailanalytics.bronze

Read Metadata

In [0]:
config_df = (
    spark.table("retailanalytics.config.table_config")
    .filter(col("ActiveFlag") == "Y")
)

display(config_df)

TableName,SourceSystem,SourceFile,SourceFormat,LoadType,SCDType,ActiveFlag,BusinessKey
products,github,products.csv,csv,FULL,NA,Y,ProductID
customers,adls,customers.csv,csv,INCREMENTAL,SCD2,Y,CustomerID
orders,adls,orders.csv,csv,INCREMENTAL,NA,Y,OrderID
exchange_rates,api,exchange_rates.json,json,FULL,NA,Y,currency_code


Bronze Load

In [0]:
for row in config_df.collect():

    table_name = row.TableName
    source_system = row.SourceSystem
    source_file = row.SourceFile
    source_format = row.SourceFormat.lower()

    source_path = (
        f"{base_path}"
        f"landing/"
        f"{source_system}/"
        f"{source_file}"
    )

    target_table = (
        f"retailanalytics.bronze.{table_name}"
    )

    print("="*60)
    print(f"Loading : {table_name}")
    print(f"Path    : {source_path}")
    print("="*60)

    if source_format == "json":

        df = (
            spark.read
            .format("json")
            .option("multiline","true")
            .option(
                f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
                storage_account_key
            )
            .load(source_path)
        )

    else:

        df = (
            spark.read
            .format(source_format)
            .option("header","true")
            .option(
                f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
                storage_account_key
            )
            .load(source_path)
        )

    bronze_df = (
        df
        .withColumn(
            "_AdfPipelineRunId",
            lit(PIPELINE_RUN_ID)
        )
        .withColumn(
            "_IngestionTimestamp",
            current_timestamp()
        )
    )

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema","true")
        .saveAsTable(target_table)
    )

    print(f"Loaded {table_name}")

Loading : products
Path    : abfss://raw@hexawaretrainingadl.dfs.core.windows.net/landing/github/products.csv
Loaded products
Loading : customers
Path    : abfss://raw@hexawaretrainingadl.dfs.core.windows.net/landing/adls/customers.csv
Loaded customers
Loading : orders
Path    : abfss://raw@hexawaretrainingadl.dfs.core.windows.net/landing/adls/orders.csv
Loaded orders
Loading : exchange_rates
Path    : abfss://raw@hexawaretrainingadl.dfs.core.windows.net/landing/api/exchange_rates.json
Loaded exchange_rates


In [0]:
for row in config_df.collect():

    table_name = row.TableName

    count = (
        spark.table(
            f"retailanalytics.bronze.{table_name}"
        ).count()
    )

    print(f"{table_name} : {count}")

products : 500
customers : 1000
orders : 5000
exchange_rates : 1


In [0]:
for row in config_df.collect():

    table_name = row.TableName

    display(
        spark.table(
            f"retailanalytics.bronze.{table_name}"
        )
    )

ProductID,ProductName,Category,SubCategory,Brand,CostPrice,_AdfPipelineRunId,_IngestionTimestamp
1,Laptop_1,Electronics,Laptop,Sony,252.44,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:05.548Z
2,Jacket_2,Fashion,Jacket,Samsung,679.93,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:05.548Z
3,Tablet_3,Electronics,Tablet,Nike,41.46,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:05.548Z
4,Laptop_4,Electronics,Laptop,Apple,510.3,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:05.548Z
5,Tablet_5,Electronics,Tablet,Apple,718.86,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:05.548Z
6,Football_6,Sports,Football,Nike,593.37,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:05.548Z
7,Laptop_7,Electronics,Laptop,Nike,346.85,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:05.548Z
8,Shirt_8,Fashion,Shirt,Sony,111.19,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:05.548Z
9,Football_9,Sports,Football,Sony,849.02,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:05.548Z
10,Chair_10,Home,Chair,Nike,540.87,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:05.548Z


CustomerID,FirstName,LastName,Email,Phone,City,State,LastUpdated,_AdfPipelineRunId,_IngestionTimestamp
1,Arjun,Patel,customer1@gmail.com,9927622808,Mumbai,Maharashtra,2026-06-20T09:49:45.109Z,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:11.157Z
2,Arjun,Iyer,customer2@gmail.com,9885934963,Mumbai,Maharashtra,2026-06-20T09:49:45.109Z,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:11.157Z
3,Anjali,Patel,customer3@gmail.com,9959015427,Delhi,Delhi,2026-06-20T09:49:45.109Z,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:11.157Z
4,Rahul,Singh,customer4@gmail.com,9186064385,Chennai,Tamil Nadu,2026-06-20T09:49:45.109Z,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:11.157Z
5,Divya,Reddy,customer5@gmail.com,9807398273,Chennai,Tamil Nadu,2026-06-20T09:49:45.109Z,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:11.157Z
6,Karthik,Reddy,customer6@gmail.com,9432222084,Bangalore,Karnataka,2026-06-20T09:49:45.109Z,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:11.157Z
7,Karthik,Sharma,customer7@gmail.com,9302502246,Bangalore,Karnataka,2026-06-20T09:49:45.109Z,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:11.157Z
8,Anjali,Reddy,customer8@gmail.com,9996434071,Mumbai,Maharashtra,2026-06-20T09:49:45.109Z,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:11.157Z
9,Karthik,Patel,customer9@gmail.com,9696355850,Mumbai,Maharashtra,2026-06-20T09:49:45.109Z,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:11.157Z
10,Priya,Patel,customer10@gmail.com,9796367641,Bangalore,Karnataka,2026-06-20T09:49:45.109Z,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:11.157Z


OrderID,CustomerID,ProductID,OrderDate,Quantity,UnitPrice,StoreCode,_AdfPipelineRunId,_IngestionTimestamp
1,924,105,2025-01-12,1,757.1,HYD001,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:16.797Z
2,709,226,2025-01-03,2,3607.22,MUM001,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:16.797Z
3,450,236,2025-03-13,2,1004.64,CHN001,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:16.797Z
4,722,359,2025-03-18,4,1787.2,CHN001,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:16.797Z
5,446,118,2025-04-29,4,1507.35,CHN001,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:16.797Z
6,884,67,2025-02-05,4,2483.76,BLR001,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:16.797Z
7,500,420,2025-04-09,1,2856.82,DEL001,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:16.797Z
8,666,403,2025-06-22,3,3978.57,BLR001,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:16.797Z
9,667,122,2025-05-19,1,2921.48,HYD001,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:16.797Z
10,482,239,2025-02-13,5,4974.21,HYD001,6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:16.797Z


base,date,rates,_AdfPipelineRunId,_IngestionTimestamp
USD,2026-06-20,"List(3.67, 0.91, 0.78, 86.54)",6a4d47c0-5a9e-4157-9df6-10b268ec6405,2026-06-27T10:48:23.227Z
